### Parâmetros necessários
- Comprimento de Bjerrum:	$l_B$	m
- Comprimento de Debye:	$\kappa^{-1}$	m
- Volume excluído não eletrostático:	$\omega$ Adimensional
- Comprimento do segmento estatístico:	$b$	m
- Número de segmentos	$N$	Adimensional
- Fração de cargas positivas:	$f$	Adimensional
- Fração de cargas negativas:	$g$	Adimensional

**Parâmetro adimensional para o volume físico**
$$
\omega^* =
\omega+
\frac{4\pi l_B(f-g)^2}{b^3\kappa^2}
-
\frac{\pi l_B^2(f+g)^2}{b^3\kappa}
$$

**Comprimento de Bjerrum**
$$
l_B=\frac{e^2}{4\pi\varepsilon_0\varepsilon_r k_BT}
$$

**Comprimento de Debye**

Para uma solução com força iônica $I$, expressa em mol/L:
$$
\kappa^2 =
\frac{2e^2N_A(1000)I}
{\varepsilon_0\varepsilon_r k_BT}
$$

**Fator de expansão** <br>
Utilizado para calcular como a cadeia se expande ou se compacta em relação à sua conformação ideal.
$$
\alpha^5-\alpha^3 =
\frac{4}{3}
\left(\frac{3}{2\pi}\right)^{3/2}
N^{1/2}\omega^*
$$

### Implementação 

In [ ]:
import numpy as np
from scipy.optimize import brentq

# FUNÇÕES AUXILIARES
def bjerrum_length(T, epsilon_r):
    """Comprimento de Bjerrum (m)."""
    return e**2 / (4 * np.pi * epsilon_0 * epsilon_r * k_B * T)

def debye_length(T, epsilon_r, ionic_strength_M):
    """Comprimento de Debye (m) e kappa (1/m)."""
    kappa_sq = (2 * e**2 * N_A * 1000 * ionic_strength_M) / (epsilon_0 * epsilon_r * k_B * T)
    kappa = np.sqrt(kappa_sq)
    return 1/kappa, kappa

# MODELO DE JOANNY (1999): ADSORÇÃO E INVERSÃO DE CARGA
def joanny_adsorption_layer(
    T=298.15,
    epsilon_r=78.5,          # Água
    ionic_strength_M=0.01,   # 10 mM (força iônica típica)
    #sigma_surface_nm2=,  
    #b_nm=,           
    #N=,               
    #f=                   
):
    """
    Calcula a carga sobrecompensada e a espessura de uma camada adsorvida
    de polieletrólito (PPy) sobre uma superfície carregada (MoS2),
    seguindo o modelo de campo médio de Joanny (1999).
    
    Retorna um dicionário com os resultados e o regime identificado.
    """
    
    # --- Conversão de unidades ---
    b = b_nm * 1e-9                # m
    sigma_nm2 = sigma_surface_nm2  # cargas/nm² (para output)
    sigma_m2 = sigma_nm2 * 1e18    # cargas/m²
    
    # --- Comprimentos eletrostáticos ---
    l_B = bjerrum_length(T, epsilon_r)
    lambda_D, kappa = debye_length(T, epsilon_r, ionic_strength_M)
    
    # --- Critério de regime ---
    # Joanny define a transição pela comparação entre a espessura da camada
    # e o comprimento de Debye. Usamos a espessura de uma cadeia ideal
    # (b*sqrt(N)) como escala de referência inicial.
    D_est = b * np.sqrt(N)
    kappa_D_est = kappa * D_est
    
    # O artigo de Joanny não fornece um critério numérico exato para kappa*D,
    # mas a física indica: kappa*D << 1 => baixa força iônica.
    # Usamos kappa*D_est < 1 como aproximação razoável.
    
    if kappa_D_est < 1.0:
        # REGIME DE BAIXA FORÇA IÔNICA (Screening Fraco)
        regime = "Baixa Força Iônica (Screening Fraco)"
        
        # Espessura da camada adsorvida: ordem da espessura de uma única
        # cadeia adsorvida. Joanny (1999) indica que D ~ R_ads.
        # para uma cadeia fracamente carregada adsorvida:
        # D ~ b * N^(1/2) * (f * l_B / b)^(1/3) (estimativa de escala)
        D = b * np.sqrt(N) * (f * l_B / b)**(1/3)
        
        # Carga sobrecompensada: proporcional a kappa (Joanny 1999)
        # sigma_over ~ sigma_surface * (1 + C * kappa * D)
        # O fator C é uma constante de ordem 1; usamos 0.5 como estimativa
        # conservadora para evitar superestimação.
        sigma_over_m2 = sigma_m2 * (1 + 0.5 * kappa * D)
        
    else:
        # REGIME DE ALTA FORÇA IÔNICA (Screening Forte)
        regime = "Alta Força Iônica (Screening Forte)"
        
        # interação eletrostática fortemente blindada
        # espessura reduzida pelo screening:
        # D ~ b * N^(1/2) / (kappa * b * N^(1/2))^(2/3)
        D = b * np.sqrt(N) / (kappa * b * np.sqrt(N))**(2/3)
        
        # Carga sobrecompensada: proporcional à carga nua da superfície.
        # Joanny (1999): sigma_over ~ sigma_surface
        sigma_over_m2 = sigma_m2
    
    # --- Conversão de volta para unidades convenientes ---
    D_nm = D * 1e9
    sigma_over_nm2 = sigma_over_m2 * 1e-18
    
    # --- Fator de sobrecompensação ---
    sobrecompensacao = sigma_over_nm2 / sigma_nm2
    
    return {
        "Regime": regime,
        "Carga Superfície (cargas/nm²)": sigma_nm2,
        "Carga Sobrecompensada (cargas/nm²)": sigma_over_nm2,
        "Fator de Sobrecompensação": sobrecompensacao,
        "Espessura da Camada (nm)": D_nm,
        "Comprimento de Debye (nm)": lambda_D * 1e9,
        "Bjerrum (nm)": l_B * 1e9,
        "kappa * D_est (critério)": kappa_D_est
    }

# Modelagem do LbL (PPy / MoS2)
def simular_multicamadas_lbl(
    n_camadas=5,
    T=298.15,
    epsilon_r=78.5,
    ionic_strength_M=0.01,
    #sigma_inicial_nm2=0.1?,   # Carga inicial do substrato MoS2
    #b_nm_ppy=,
    #N_ppy=,
    #f_ppy=
):
    """
    Simula a deposição sequencial de camadas de PPy e MoS2.
    
    A primeira camada é de PPy sobre MoS2 (assumido carregado negativamente).
    A carga sobrecompensada da camada 1 se torna a nova carga superficial
    para a camada 2 (com sinal oposto), e assim por diante.
    """
    
    resultados = []
    sigma_atual = sigma_inicial_nm2  # Carga da superfície atual
    sinal_atual = -1  # MoS2 assumido negativo
    
    for i in range(n_camadas):
        # Determina o material da camada
        if i % 2 == 0:
            material = "PPy"
            # PPy é positivo (dopado), adsorve sobre superfície negativa
            # o modelo de Joanny assume polieletrólito de carga oposta à superfície
            # como o MoS2 é de carga negativa, achei válido dopar o PPy
            if sinal_atual >= 0:
                # Mas se a superfície for positiva, invertemos o sinal de f
                pass
        else:
            material = "MoS2"
            # MoS2 (funcionalizado ou com carga residual) adsorve sobre PPy positivo
            # (simplificação: usamos a mesma magnitude da carga inicial)
        
        # Para simplificar, aplicamos o modelo de Joanny apenas para o PPy.
        # Para o MoS2, assumimos uma adsorção que inverte a carga para o valor
        # da carga inicial do substrato (aproximação).
        
        if material == "PPy":
            # Usa o modelo de Joanny para calcular a sobrecompensação
            res = joanny_adsorption_layer(
                T=T,
                epsilon_r=epsilon_r,
                ionic_strength_M=ionic_strength_M,
                sigma_surface_nm2=sigma_atual,
                b_nm=b_nm_ppy,
                N=N_ppy,
                f=f_ppy
            )
            sigma_nova = res["Carga Sobrecompensada (cargas/nm²)"]
            espessura = res["Espessura da Camada (nm)"]
            regime = res["Regime"]
        else:
            # MoS2: simplificação - assume que a camada de MoS2 adsorve
            # e a carga superficial retorna à magnitude da carga inicial,
            # mas com sinal invertido (comportamento típico de LbL).
            # Em um modelo mais refinado, seria necessário o modelo de Joanny
            # para o MoS2 também (com seus próprios parâmetros).
            sigma_nova = sigma_inicial_nm2 * 1.0  # Magnitude similar
            espessura = 1.0  # Espessura típica de nanosheet de MoS2 (nm, aproximado)
            regime = "Adsorção de Nanosheet (Simplificado)"
        
        # Armazena resultados
        resultados.append({
            "Camada": i + 1,
            "Material": material,
            "Carga Superfície Antes (cargas/nm²)": sigma_atual,
            "Carga Após Adsorção (cargas/nm²)": sigma_nova,
            "Espessura (nm)": espessura,
            "Regime": regime
        })
        
        # Atualiza o estado para a próxima camada
        # A carga da superfície se inverte (com a magnitude da sobrecompensação)
        sigma_atual = sigma_nova
        sinal_atual = 1 if material == "PPy" else -1
    
    return resultados

# (im)Print 
if __name__ == "__main__":
    print("=" * 70)
    print("MODELO DE JOANNY (1999) - ADSORÇÃO E INVERSÃO DE CARGA")
    print("Sistema: Layer-by-Layer de PPy / MoS2")
    print("=" * 70)
    
    # --- Parâmetros do sistema ---
    params = {
        "T": 298.15,
        "epsilon_r": 78.5,
        "ionic_strength_M": 0.01,    # 10 mM
        "sigma_inicial_nm2": 0.1,    # Carga estimada do MoS2
        "b_nm_ppy": 0.7,
        "N_ppy": 100,
        "f_ppy": 0.3
    }
    
    print("\nParâmetros Utilizados:")
    for k, v in params.items():
        print(f"  {k}: {v}")
    
    # --- Simulação de uma única camada de PPy sobre MoS2 ---
    print("\n" + "-" * 70)
    print("CAMADA ÚNICA: PPy sobre MoS2")
    print("-" * 70)
    
    resultado_unico = joanny_adsorption_layer(
        T=params["T"],
        epsilon_r=params["epsilon_r"],
        ionic_strength_M=params["ionic_strength_M"],
        sigma_surface_nm2=params["sigma_inicial_nm2"],
        b_nm=params["b_nm_ppy"],
        N=params["N_ppy"],
        f=params["f_ppy"]
    )
    
    for chave, valor in resultado_unico.items():
        if isinstance(valor, float):
            print(f"  {chave}: {valor:.4f}")
        else:
            print(f"  {chave}: {valor}")
    
    # Simulação do Layer by Layer
    print("\n" + "-" * 70)
    print("Simulação do Layer by Layer (5 Camadas)")
    print("-" * 70)
    
    camadas = simular_multicamadas_lbl(
        n_camadas=5,
        **params
    )
    
    print(f"\n{'Camada':<8} {'Material':<10} {'Carga Antes':<14} {'Carga Depois':<14} {'Espessura (nm)':<16} {'Regime'}")
    print("-" * 85)
    for c in camadas:
        print(f"{c['Camada']:<8} {c['Material']:<10} "
              f"{c['Carga Superfície Antes (cargas/nm²)']:<14.4f} "
              f"{c['Carga Após Adsorção (cargas/nm²)']:<14.4f} "
              f"{c['Espessura (nm)']:<16.2f} {c['Regime']}")
    
    # --- Análise do regime ---
    print("\n" + "-" * 70)
    print("ANÁLISE DO REGIME (PPy sobre MoS2)")
    print("-" * 70)
    
    kappa_D = resultado_unico["kappa * D_est (critério)"]
    print(f"  kappa * D_est = {kappa_D:.4f}")
    if kappa_D < 1:
        print("  → Regime de BAIXA força iônica: sobrecompensação forte.")
        print("    A carga sobrecompensada é proporcional a kappa (Joanny 1999).")
    else:
        print("  → Regime de ALTA força iônica: screening forte.")
        print("    A carga sobrecompensada é proporcional à carga nua da superfície.")
    
    print("\n" + "=" * 70)
    print("FIM DA SIMULAÇÃO")
    print("=" * 70)